# TensorFlow, convolutional networks, and observable training

**Applied question.** How can we build an image classifier whose computation we understand and whose training behavior we can inspect? We begin with handwritten digits, expose what a convolution actually computes, and finish with a defensible cats-versus-dogs pathway.

## How to use this notebook

Run the **core** cells in order. On the normal student route, the first run needs internet access: `tf.keras.datasets.mnist.load_data()` downloads the genuine full MNIST archive and stores it in Keras's user cache. Later runs reuse that cache. Continuous integration sets `RICE_DSM_RUNNING_NOTEBOOK_TESTS=1` and selects the repository's small, versioned, authentic-MNIST sample. That bounded route tests software behavior; it is **not** the scientific experiment or a substitute for analyzing the full dataset.

Before each code cell, state its input shape, output shape, and purpose. **Practice** cells ask you to predict before executing. **Extension** work belongs on a branch with tests and recorded evidence.

## Learning objectives

By the end, you should be able to:

1. distinguish TensorFlow tensors and autodiff, the Keras model API, and TensorBoard observability;
2. explain every transition in `acquire → validate → transform → build → compile → fit → evaluate → predict → save`;
3. train and diagnose a dense classifier on the full official MNIST dataset while explaining why CI uses a bounded real-data sample;
4. calculate kernel, stride, padding, channel, pooling, parameter-count, and receptive-field geometry; and
5. choose training, validation, data, and system metrics that make failure diagnosable without leaking sensitive or high-cardinality data.

## Kernel preflight: verify the process before the experiment

A notebook kernel is a long-running Python process. VS Code can remember a kernel from another project even after you open this repository. The next cell uses only Python's standard library, reports the interpreter actually running, and stops with exact recovery commands before importing TensorFlow.

In [ ]:
import os
import sys
from pathlib import Path

root_hint = os.environ.get("RICE_DSM_PROJECT_ROOT")
search_roots = ([Path(root_hint)] if root_hint else []) + [Path.cwd(), *Path.cwd().parents]
course_root = next((path.resolve() for path in search_roots if (path / "pyproject.toml").is_file()), None)
if course_root is None:
    raise RuntimeError("Open the course repository folder in VS Code, then restart this notebook.")
expected_prefix = (course_root / ".venv").resolve()
actual_prefix = Path(sys.prefix).resolve()
print("Python executable:", sys.executable)
print("Active environment:", actual_prefix)
print("Expected environment:", expected_prefix)
if actual_prefix != expected_prefix:
    raise RuntimeError(
        "Wrong notebook kernel. In the repository terminal run:\n"
        "  uv sync --locked\n"
        "  uv run python scripts/setup_course.py\n"
        "Then select 'Rice DSM' in VS Code and restart the kernel."
    )
assert Path(sys.executable).is_file()

## Why this matters: from dataflow systems to a high-level learning API

TensorFlow was open-sourced in November 2015 as an interface and implementation for machine-learning computations across heterogeneous devices. TensorFlow 2 made eager execution the default and standardized on Keras as its high-level API. Modern Keras is also a multi-backend API; in this lesson we use its TensorFlow backend.

Keep the layers of the stack distinct:

- **`tf.Tensor` / operations:** typed multidimensional arrays and kernels on CPU, GPU, or accelerator.
- **`tf.Variable` + `tf.GradientTape`:** mutable parameters and reverse-mode automatic differentiation.
- **Keras layers and models:** composable parameterized transformations.
- **`compile`, `fit`, `evaluate`, `predict`:** a managed training and inference lifecycle.
- **callbacks:** hooks for checkpoints, early stopping, schedules, and telemetry.
- **TensorBoard:** a separate application that reads event logs; it is evidence tooling, not the optimizer.

Primary sources: [TensorFlow 2015 white paper](https://www.tensorflow.org/static/extras/tensorflow-whitepaper2015.pdf), [TensorFlow basics](https://www.tensorflow.org/guide/basics), [automatic differentiation](https://www.tensorflow.org/guide/autodiff), [Keras guide](https://www.tensorflow.org/guide/keras), and [Keras 3 overview](https://keras.io/about/).

In [ ]:
import importlib.util
import logging
import os

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyBboxPatch

from rice_dsm.paths import mnist_teaching_sample_path

# One named seed makes the demonstration repeatable enough to debug. It does
# not imply that all devices or TensorFlow releases are bitwise identical.
SEED = 438

# CI selects a deliberately different execution budget. This flag must never
# be used to describe CI metrics as scientific results.
CI_SMOKE_TEST = os.environ.get("RICE_DSM_RUNNING_NOTEBOOK_TESTS") == "1"
SAMPLE_PATH = mnist_teaching_sample_path()

# TensorBoard writes event files beneath one stable repository-relative root.
LOG_ROOT = course_root / "output" / "tensorboard" / "lecture-12"
RUN_NAME = "mnist-dense-seed-438"
LOG_DIR = LOG_ROOT / RUN_NAME

np.random.seed(SEED)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    force=True,
)
logger = logging.getLogger("rice_dsm.lecture12.tensorflow")

TF_AVAILABLE = importlib.util.find_spec("tensorflow") is not None
TENSORBOARD_AVAILABLE = importlib.util.find_spec("tensorboard") is not None
RUN_TENSORFLOW = TF_AVAILABLE
logger.info(
    "TensorFlow installed=%s; TensorBoard installed=%s; CI smoke test=%s",
    TF_AVAILABLE,
    TENSORBOARD_AVAILABLE,
    CI_SMOKE_TEST,
)

assert SEED == 438
assert logger.name.endswith("tensorflow")
assert SAMPLE_PATH.is_file()
assert TENSORBOARD_AVAILABLE, "Run `uv sync --locked` to install TensorBoard."


## Tensors and autodiff: the bridge from our manual network

A tensor has a **shape**, **dtype**, **device**, and value. A rank-two image batch has shape `(batch, height, width)`; a color-image batch usually has shape `(batch, height, width, channels)`. TensorFlow tensors are immutable, while `tf.Variable` stores trainable state.

Inside a gradient tape, TensorFlow records differentiable operations. For scalar loss $L$ and variables $\theta$, reverse mode computes $\nabla_\theta L$ by applying the same chain rule we derived manually. It automates derivative bookkeeping; it does not select the target, split the data, prevent leakage, or decide whether a metric represents real cost.

In [ ]:
# Forward calculation: y_hat = x^T w, then L = 1/2 (y_hat - y)^2.
# Shapes: x and w are (3,), while prediction and loss are scalars.
x_numpy = np.array([1.0, 2.0, 3.0])
w_numpy = np.array([0.2, -0.1, 0.4])
target = 1.5
prediction = x_numpy @ w_numpy

# Chain rule: dL/dw = (y_hat - y) x.
gradient_by_hand = (prediction - target) * x_numpy

if RUN_TENSORFLOW:
    import tensorflow as tf

    tf.random.set_seed(SEED)
    x = tf.constant(x_numpy, dtype=tf.float32)  # observed input: immutable
    w = tf.Variable(w_numpy, dtype=tf.float32)  # trainable state: watched
    with tf.GradientTape() as tape:
        # The tape records the tensor operations needed for reverse mode.
        loss = 0.5 * (tf.tensordot(x, w, axes=1) - target) ** 2
    gradient_by_tape = tape.gradient(loss, w).numpy()
    np.testing.assert_allclose(gradient_by_tape, gradient_by_hand, rtol=1e-5)
else:
    gradient_by_tape = gradient_by_hand.copy()
    logger.info("Using the derived gradient because TensorFlow is unavailable.")

assert gradient_by_tape.shape == (3,)
assert np.isfinite(gradient_by_tape).all()
gradient_by_tape


## Worked example: MNIST in the Keras lifecycle

MNIST contains 28-by-28 grayscale handwritten digits. A dense model first flattens each image from shape $(28,28)$ to a vector in $\mathbb{R}^{784}$. Flattening is a reshape, not feature learning: nearby pixels become merely adjacent coordinates. Pixel intensities are scaled from integers in $[0,255]$ to floating-point values in $[0,1]$.

For ten mutually exclusive classes, a model emits ten logits $z_k$. Sparse categorical cross-entropy accepts integer labels; categorical cross-entropy expects one-hot vectors $e_y$. Both encode the same class identity when configured correctly. A fused logits loss computes

$$\ell(z,y)=-z_y+\log\left(\sum_{k=0}^{9}e^{z_k}\right).$$

Keras makes the training call short, but the scientific workflow is not one indivisible operation. The next five steps separate acquisition, validation and transformation, model construction and compilation, fitting and selection, and final evaluation. The test set remains untouched until the last step.

## Step 1 — acquire data without hiding provenance

The two routes below are intentionally explicit.

- **Normal lecture/student route:** `tf.keras.datasets.mnist.load_data()` obtains the full canonical train/test split. Keras downloads `mnist.npz` only when it is absent from its user cache (normally `~/.keras/datasets/`). The cache is convenient, but it is not a dataset-versioning strategy for a production project. Record source, checksum/version, license, and split semantics in real work.
- **CI smoke-test route:** the environment variable selects a small repository fixture made from genuine MNIST observations. It lets automation test loading, shapes, training, evaluation, and logging quickly and without network access. Metrics from this sample must never be reported as the course experiment.

The variable names begin with `raw_` because acquisition has not yet changed dtype, numerical range, or shape.

In [ ]:
if RUN_TENSORFLOW:
    if CI_SMOKE_TEST:
        # Automation uses a bounded, versioned subset so CI is fast and offline.
        with np.load(SAMPLE_PATH) as mnist:
            raw_train_images = mnist["x_train"]
            y_train = mnist["y_train"]
            raw_test_images = mnist["x_test"]
            y_test = mnist["y_test"]
        data_source = "versioned authentic-MNIST CI sample (software evidence only)"
    else:
        # The normal route downloads once and then reuses the Keras user cache.
        (raw_train_images, y_train), (raw_test_images, y_test) = (
            tf.keras.datasets.mnist.load_data()
        )
        data_source = "full official MNIST via tf.keras.datasets"
else:
    # These placeholders keep the non-TensorFlow explanatory route explicit.
    raw_train_images = raw_test_images = y_train = y_test = None
    data_source = "TensorFlow unavailable; training sections are skipped"

logger.info("data_source=%s", data_source)
if RUN_TENSORFLOW:
    print(
        data_source,
        {
            "train_images": raw_train_images.shape,
            "train_labels": y_train.shape,
            "test_images": raw_test_images.shape,
            "test_labels": y_test.shape,
        },
    )
    assert raw_train_images.shape[1:] == (28, 28)
    assert raw_test_images.shape[1:] == (28, 28)
    assert raw_train_images.dtype == np.uint8

In [ ]:
def first_examples_per_class(
    labels: np.ndarray, per_class: int
) -> np.ndarray:
    """Return deterministic, class-interleaved indices."""
    by_class = [
        np.flatnonzero(labels == digit)[:per_class] for digit in range(10)
    ]
    # Interleaving keeps the final validation fraction from containing one class.
    return np.stack(by_class, axis=1).reshape(-1)


if RUN_TENSORFLOW and CI_SMOKE_TEST:
    # CI asks whether the complete path still works, not how accurate it is.
    # Sixteen train and four test images per digit bound TensorFlow runtime
    # while exercising all ten output classes.
    ci_train_indices = first_examples_per_class(y_train, per_class=16)
    ci_test_indices = first_examples_per_class(y_test, per_class=4)
    raw_train_images = raw_train_images[ci_train_indices]
    y_train = y_train[ci_train_indices]
    raw_test_images = raw_test_images[ci_test_indices]
    y_test = y_test[ci_test_indices]
    logger.info("CI micro-slice: train=%d test=%d", len(y_train), len(y_test))

assert not CI_SMOKE_TEST or not RUN_TENSORFLOW or len(y_train) == 160
assert not CI_SMOKE_TEST or not RUN_TENSORFLOW or len(y_test) == 40


## Step 2 — validate first, then transform

An image tensor entering the dense baseline has shape `(observation, height, width)`. TensorFlow's `Flatten` layer will later turn each `(28, 28)` image into 784 features. We keep that reshape inside the model so the saved model owns the shape transformation.

We do perform a numerical transformation at the data boundary: convert `uint8` pixels in $[0,255]$ to `float32` in $[0,1]$. Division after casting avoids integer arithmetic and `float32` matches the framework's normal training dtype. This scaling uses a fixed physical bound rather than a statistic estimated from the data, so there is no fitted preprocessing state to leak across the split.

The assertions are executable data contracts. They should fail before training, where the cause is still local and understandable.

In [ ]:
if RUN_TENSORFLOW:
    # Transform images without mutating the raw arrays used for inspection.
    x_train = raw_train_images.astype(np.float32) / 255.0
    x_test = raw_test_images.astype(np.float32) / 255.0
    y_train = y_train.astype(np.int64)
    y_test = y_test.astype(np.int64)

    # Validate representation, vocabulary, and train/test alignment.
    assert x_train.ndim == 3 and x_train.shape[1:] == (28, 28)
    assert x_test.ndim == 3 and x_test.shape[1:] == (28, 28)
    assert x_train.dtype == np.float32 and x_test.dtype == np.float32
    assert 0.0 <= float(x_train.min()) <= float(x_train.max()) <= 1.0
    assert set(np.unique(y_train)) == set(range(10))
    assert len(x_train) == len(y_train) and len(x_test) == len(y_test)

    fig, axes = plt.subplots(2, 5, figsize=(8, 3.8), constrained_layout=True)
    for digit, axis in enumerate(axes.flat):
        example_index = int(np.flatnonzero(y_train == digit)[0])
        axis.imshow(x_train[example_index], cmap="gray_r", vmin=0.0, vmax=1.0)
        axis.set_title(f"label {digit}")
        axis.axis("off")
    fig.suptitle("One training observation from each MNIST class", fontweight="bold")
    plt.show()

## Step 3 — build the computation, then compile the learning contract

`Sequential` is appropriate because this graph is a single stack with one input and one output. Building defines the forward map:

1. `Input((28, 28))` declares the shape of one observation; the batch dimension remains `None`.
2. `Flatten()` maps `(batch, 28, 28)` to `(batch, 784)` without trainable parameters.
3. `Dense(64, relu)` learns a hidden representation.
4. `Dense(10)` emits one **logit** per digit. It deliberately omits softmax because the fused cross-entropy loss receives logits more stably.

Compilation does not train. It binds three separate policies: Adam determines parameter updates, cross-entropy supplies the optimized scalar objective, and accuracy supplies human-readable monitoring. Changing any one changes the experiment.

In [ ]:
if RUN_TENSORFLOW:
    # Seed Python/NumPy/TensorFlow together, then request deterministic kernels.
    tf.keras.utils.set_random_seed(SEED)
    tf.config.experimental.enable_op_determinism()

    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(28, 28), name="pixels"),
            tf.keras.layers.Flatten(name="flatten_pixels"),
            tf.keras.layers.Dense(64, activation="relu", name="hidden_features"),
            tf.keras.layers.Dense(10, name="digit_logits"),
        ],
        name="mnist_dense_baseline",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )

    # The normal route writes scalar event records. CI omits the callback so
    # framework startup and filesystem telemetry cannot dominate a smoke test.
    telemetry = None
    if not CI_SMOKE_TEST:
        telemetry = tf.keras.callbacks.TensorBoard(
            log_dir=str(LOG_DIR),
            histogram_freq=0,  # scalar-only core run: smaller and faster
            update_freq="epoch",
            profile_batch=0,  # profiling is opt-in because instrumentation has cost
        )

    assert model.input_shape == (None, 28, 28)
    assert model.output_shape == (None, 10)
    assert model.count_params() == 50_890

## Step 4 — fit on training data, select with validation data

`fit` updates parameters. With `validation_split=0.1`, Keras reserves the last 10% of the supplied training arrays before training; those observations receive no gradient updates. We use `shuffle=False` so the bounded CI route and the lecture route follow a transparent deterministic ordering. In a research study, define and persist the validation split explicitly—especially when observations have groups, time order, or repeated subjects.

The normal full-data route uses two epochs and writes TensorBoard events. CI uses one epoch, a 160-image class-balanced training micro-slice, and no TensorBoard callback. Both routes exercise the same model API, but only the normal route supports empirical discussion. The `History` object mirrors the scalar series written by the normal route.

In [ ]:
# Execution budgets are configuration, not hidden control flow.
FIT_EPOCHS = 1 if CI_SMOKE_TEST else 2
FIT_CALLBACKS = [] if CI_SMOKE_TEST or not RUN_TENSORFLOW else [telemetry]

# This cheap assertion verifies log-path and dependency configuration even
# when CI deliberately skips TensorBoard event writing.
assert LOG_DIR.parent == LOG_ROOT
assert TENSORBOARD_AVAILABLE
assert not CI_SMOKE_TEST or FIT_CALLBACKS == []


In [ ]:
history = None
if RUN_TENSORFLOW:
    history = model.fit(
        x_train,
        y_train,
        validation_split=0.1,
        epochs=FIT_EPOCHS,
        batch_size=128,
        shuffle=False,
        callbacks=FIT_CALLBACKS,
        verbose=0,
    )

    recorded_metrics = set(history.history)
    assert {"loss", "accuracy", "val_loss", "val_accuracy"} <= recorded_metrics
    print({name: values[-1] for name, values in history.history.items()})

## Step 5 — freeze choices, evaluate once, and inspect errors

`evaluate` runs the frozen model on the untouched official test partition; it does not update weights. `predict` returns logits, not probabilities or labels. `argmax` converts each ten-logit vector to the highest-scoring class. If a downstream decision needs probabilities, apply softmax deliberately and assess calibration rather than assuming the scores are trustworthy probabilities.

The confusion matrix asks a better question than accuracy alone: *which true digits become which predicted digits?* Rows below are true classes and columns are predicted classes. The CI matrix is only a pipeline check; only the full-data route supports the lecture's empirical discussion.

In [ ]:
mnist_result = None
if RUN_TENSORFLOW:
    test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
    test_logits = model.predict(x_test, verbose=0)
    predicted_labels = test_logits.argmax(axis=1)

    confusion = np.zeros((10, 10), dtype=int)
    np.add.at(confusion, (y_test, predicted_labels), 1)
    mnist_result = {
        "data_source": data_source,
        "test_loss": float(test_loss),
        "test_accuracy": float(test_accuracy),
        "test_observations": int(len(y_test)),
    }
    logger.info("MNIST final test metrics: %s", mnist_result)

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
    epochs = np.arange(1, len(history.history["loss"]) + 1)
    axes[0].plot(epochs, history.history["loss"], marker="o", label="train")
    axes[0].plot(epochs, history.history["val_loss"], marker="o", label="validation")
    axes[0].set(title="Cross-entropy", xlabel="epoch", ylabel="loss")
    axes[0].legend()
    axes[1].plot(epochs, history.history["accuracy"], marker="o", label="train")
    axes[1].plot(epochs, history.history["val_accuracy"], marker="o", label="validation")
    axes[1].set(title="Accuracy", xlabel="epoch", ylabel="fraction correct")
    axes[1].legend()
    image = axes[2].imshow(confusion, cmap="Blues")
    axes[2].set(title="Held-out confusion counts", xlabel="predicted digit", ylabel="true digit")
    axes[2].set_xticks(range(10)); axes[2].set_yticks(range(10))
    fig.colorbar(image, ax=axes[2], fraction=0.046)
    plt.show()
else:
    logger.info("TensorFlow unavailable; the mathematics and diagrams still execute.")

assert mnist_result is None or 0.0 <= mnist_result["test_accuracy"] <= 1.0

### Guided checkpoint: read the model as a contract

Before running the next cell, answer: Why does the last layer omit softmax? What changes if labels are one-hot? Which line creates a validation partition, and why is the test set still not validation data?

In [ ]:
def one_hot(labels: np.ndarray, class_count: int) -> np.ndarray:
    """Encode integer class labels as rows of indicator vectors."""
    labels = np.asarray(labels, dtype=int)
    if labels.ndim != 1 or np.any((labels < 0) | (labels >= class_count)):
        raise ValueError("labels must be a 1-D array in [0, class_count)")
    encoded = np.zeros((len(labels), class_count), dtype=float)
    encoded[np.arange(len(labels)), labels] = 1.0
    return encoded

example_labels = np.array([3, 0, 9])
encoded = one_hot(example_labels, 10)
assert encoded.shape == (3, 10)
assert np.all(encoded.sum(axis=1) == 1.0)
assert np.array_equal(encoded.argmax(axis=1), example_labels)
encoded

## Why convolution fits images

A dense layer forgets that pixels form a grid. A convolution preserves locality and reuses one small kernel across locations. For one input channel, kernel $K\in\mathbb{R}^{k_h\times k_w}$, and bias $b$, the cross-correlation used by deep-learning libraries is

$$z_{i,j}=b+\sum_{u=0}^{k_h-1}\sum_{v=0}^{k_w-1}K_{u,v}x_{i+u,j+v}.$$

- **Kernel/filter:** learned local pattern detector.
- **Stride:** how far the kernel moves; larger stride reduces spatial resolution.
- **Padding:** values added around the border. `valid` uses none; `same` typically preserves size for stride one.
- **Channels:** input feature planes. A color filter spans all three RGB channels; each output filter creates one output channel.
- **Pooling:** fixed local aggregation such as maximum; it reduces resolution and provides limited local invariance.
- **Receptive field:** original input region capable of influencing an activation. Stacking local layers makes the effective field grow.

For input width $n$, padding $p$, kernel width $k$, and stride $s$, the output width is

$$\left\lfloor\frac{n+2p-k}{s}\right\rfloor+1.$$

In [ ]:
def correlate2d(image: np.ndarray, kernel: np.ndarray, *, stride: int = 1, padding: int = 0) -> np.ndarray:
    """Small, inspectable single-channel correlation used only for teaching."""
    if image.ndim != 2 or kernel.ndim != 2:
        raise ValueError("image and kernel must both be rank-two")
    padded = np.pad(image, padding)
    out_h = (padded.shape[0] - kernel.shape[0]) // stride + 1
    out_w = (padded.shape[1] - kernel.shape[1]) // stride + 1
    output = np.empty((out_h, out_w), dtype=float)
    for i in range(out_h):
        for j in range(out_w):
            patch = padded[i * stride : i * stride + kernel.shape[0], j * stride : j * stride + kernel.shape[1]]
            output[i, j] = np.sum(patch * kernel)
    return output


### A fixture isolates mechanism; it is not animal data

The two 14-by-14 arrays below are deliberately schematic. They let us inspect one known vertical-edge kernel without downloading a second dataset or pretending that a toy pattern validates cats-versus-dogs classification. The kernel has shape `(3, 3)`; `padding=1` preserves the fixture's `(14, 14)` spatial shape. A positive response means the local patch becomes brighter from left to right under this chosen kernel.


In [ ]:
# A bounded offline fixture: point ears versus floppy ears, not real animals.
cat_like = np.zeros((14, 14), dtype=float)
cat_like[2:4, 3:5] = 1
cat_like[2:4, 9:11] = 1
cat_like[4:12, 3:11] = 0.65
dog_like = np.zeros((14, 14), dtype=float)
dog_like[3:11, 4:10] = 0.65
dog_like[5:10, 1:4] = 1
dog_like[5:10, 10:13] = 1
vertical_edge = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=float)
cat_edges = correlate2d(cat_like, vertical_edge, padding=1)

assert cat_edges.shape == cat_like.shape
assert correlate2d(cat_like, vertical_edge, stride=2).shape == (6, 6)
assert np.isfinite(cat_edges).all()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(11, 3), constrained_layout=True)
for axis, image, title, cmap in zip(
    axes,
    [cat_like, dog_like, vertical_edge, cat_edges],
    ["offline cat-like fixture", "offline dog-like fixture", "vertical-edge kernel", "cat feature map"],
    ["gray_r", "gray_r", "coolwarm", "coolwarm"],
    strict=True,
):
    displayed = axis.imshow(image, cmap=cmap)
    axis.set_title(title, fontsize=9)
    axis.set_xticks([]); axis.set_yticks([])
    if title == "vertical-edge kernel":
        for (row, column), value in np.ndenumerate(image):
            axis.text(column, row, f"{value:.0f}", ha="center", va="center")
fig.suptitle("A learned filter responds to local structure, not an animal label", fontweight="bold")
plt.show()

The fixture isolates mechanics and makes no claim about real-world accuracy. A production cats-versus-dogs system must confront background shortcuts, duplicate animals, breed imbalance, photographer effects, corrupted images, licensing, and distribution shift. A model may appear accurate because it learned grass versus furniture rather than dogs versus cats.

In [ ]:
fig, axis = plt.subplots(figsize=(11, 3.2))
axis.set_xlim(0, 11); axis.set_ylim(0, 3); axis.axis("off")
blocks = [
    (0.2, 0.65, 1.6, 1.7, "image\n28×28×1", "#D9EAF7"),
    (2.2, 0.65, 1.8, 1.7, "Conv 3×3\n16 maps", "#B7D8C2"),
    (4.4, 0.65, 1.5, 1.7, "MaxPool\n2×2", "#F8D8A8"),
    (6.3, 0.65, 1.8, 1.7, "Conv 3×3\n32 maps", "#B7D8C2"),
    (8.5, 0.65, 1.2, 1.7, "Pool", "#F8D8A8"),
    (10.0, 0.65, 0.8, 1.7, "logits\n10", "#E8C7D8"),
]
for x, y, width, height, label, color in blocks:
    patch = FancyBboxPatch((x, y), width, height, boxstyle="round,pad=0.04", facecolor=color, edgecolor="#19324D")
    axis.add_patch(patch); axis.text(x + width / 2, y + height / 2, label, ha="center", va="center", fontsize=9)
for left, right in zip(blocks[:-1], blocks[1:], strict=True):
    axis.annotate("", xy=(right[0] - 0.08, 1.5), xytext=(left[0] + left[2] + 0.08, 1.5), arrowprops={"arrowstyle": "->", "color": "#19324D"})
axis.set_title("A CNN trades spatial resolution for richer learned channels", fontweight="bold")
plt.show()

### The diagram becomes a Keras model

`Conv2D(filters=16, kernel_size=3, padding="same")` learns 16 kernels, each spanning the input channel dimension. The layer therefore emits 16 feature maps. `GlobalAveragePooling2D` averages each final feature map into one number and avoids a large flatten-and-dense parameter block. Keras defaults to Glorot-uniform kernel initialization for these layers; initialization is part of the experiment and should be named rather than treated as magic.

In [ ]:
if RUN_TENSORFLOW:
    cnn_model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(28, 28, 1), name="grayscale_image"),
            # Same padding preserves 28×28; 16 filters create 16 channels.
            tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
            # A 2×2 pool halves the spatial grid: 28×28 → 14×14.
            tf.keras.layers.MaxPool2D(pool_size=2),
            # The second convolution maps 16 input channels to 32 output channels.
            tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
            # The second pool yields a 7×7×32 representation.
            tf.keras.layers.MaxPool2D(pool_size=2),
            # Average each feature map, producing 32 features rather than 7×7×32.
            tf.keras.layers.GlobalAveragePooling2D(),
            tf.keras.layers.Dense(10, name="digit_logits"),
        ],
        name="mnist_small_cnn",
    )
    cnn_parameter_count = cnn_model.count_params()
    assert cnn_model.output_shape == (None, 10)
else:
    # 16*(3*3*1 + 1) + 32*(3*3*16 + 1) + 10*(32 + 1)
    cnn_parameter_count = 5_130

assert cnn_parameter_count == 5_130
cnn_parameter_count

### Receptive fields grow through composition

With stride one, two 3-by-3 convolutions give a unit an effective 5-by-5 receptive field; three give 7-by-7. Pooling and strides make it grow faster because one step in a deeper map corresponds to several input pixels. This hierarchy is the key inductive bias: early filters can detect edges, later filters combine them into parts, and still later units combine parts into task-relevant patterns. These descriptions are hypotheses to inspect—not guarantees that every channel has a human-readable meaning.

## Optional real cats-versus-dogs pathway

Use a separate branch and the official [TensorFlow transfer-learning tutorial](https://www.tensorflow.org/tutorials/images/transfer_learning) as the data/licensing reference. Do not commit the image archive. Build a manifest containing a stable sample identifier, source, label, group identifier (animal/owner when available), split, and checksum. Split by animal before augmentation. Begin with a small CNN, then compare transfer learning under the same frozen test contract.

A defensible result includes per-class precision/recall, confusion counts, calibration or confidence analysis, subgroup/error slices, representative false positives/negatives, wall-clock cost, and a model card. “The loss decreased” is not sufficient evidence.

## Professional practice: TensorBoard and logging make training observable

Python logging records discrete events and context: configuration identity, dataset counts, checkpoint path, exceptions, and final evaluation. TensorBoard records time-indexed summaries: training/validation loss, task metrics, learning rate, parameter or activation histograms, images, and—when intentionally enabled—performance traces.

The full-data route attaches a TensorBoard callback because those curves are lecture evidence. The CI route verifies that TensorBoard is importable and the log path is well formed, but attaches no callback and writes no events. Event-writer startup and filesystem work should not dominate a smoke test.

Prefer metrics that answer a question:

- **optimization:** batch/epoch loss, gradient norm, learning rate, nonfinite counts;
- **generalization:** validation loss and task metrics, plus the train–validation gap;
- **data:** class counts, corrupt records, preprocessing ranges, batch latency;
- **system:** step time, examples/second, memory/device utilization when profiling;
- **release:** one final frozen-test report, artifact identity, and inference latency.

High-cardinality identifiers and raw sensitive examples should not be logged. Training metrics are telemetry, not proof of deployment quality.

In [ ]:
# The normal full-data route wrote lightweight scalar telemetry. A longer
# study may add histograms and early stopping. CI constructs no event writer.
if RUN_TENSORFLOW and not CI_SMOKE_TEST:
    production_callbacks = [
        tf.keras.callbacks.TensorBoard(
            log_dir=str(LOG_DIR),
            histogram_freq=1,
            update_freq="epoch",
            profile_batch=0,  # enable a short range intentionally; profiling adds overhead
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True
        ),
    ]
else:
    production_callbacks = []

logger.info("run=%s seed=%d log_dir=%s", RUN_NAME, SEED, LOG_DIR)
assert LOG_DIR.parts[-1] == RUN_NAME
assert not CI_SMOKE_TEST or production_callbacks == []

### Launch TensorBoard from the repository root

In the VS Code terminal, with the project environment synchronized:

```text
uv run tensorboard --logdir output/tensorboard/lecture-12 --port 6006
```

Then open `http://localhost:6006` in a browser. Keep the terminal running; stop the local server with **Ctrl+C**. If port 6006 is occupied, choose another port such as `--port 6007`. Do not bind to a public interface or expose experiment logs without authorization.

In VS Code you may also open the Command Palette and choose **Python: Select Interpreter**, then select this repository's `.venv`. Finally choose the **Rice DSM** notebook kernel. The interpreter and the notebook kernel must identify the same environment.

See the official [TensorBoard training guide](https://www.tensorflow.org/guide/keras/training_with_built_in_methods#visualizing_loss_and_metrics_during_training), [Profiler guide](https://www.tensorflow.org/guide/profiler), and Python's [logging documentation](https://docs.python.org/3/library/logging.html).

## Debugging and common failure modes

1. **Wrong kernel:** `import tensorflow` fails although installation succeeded elsewhere. Inspect `sys.executable`; select Rice DSM.
2. **Shape mismatch:** dense MNIST expects `(batch, 28, 28)` before `Flatten`; a CNN expects an explicit channel dimension `(batch, 28, 28, 1)`.
3. **Loss/output mismatch:** `from_logits=True` requires raw logits. Do not apply softmax twice.
4. **Encoding mismatch:** sparse cross-entropy expects integer labels; categorical cross-entropy expects one-hot targets.
5. **Leakage:** augmentation, normalization estimates, and model selection must not learn from the final test set.
6. **Empty TensorBoard:** callback and command may point to different log roots, or no event has yet been written.
7. **Profiler slowdown:** trace only a short batch range. Instrumentation has cost.
8. **False confidence:** accuracy can hide class imbalance and asymmetric error costs. Inspect confusion counts and per-class metrics.

## Independent practice and extension

**Guided.** Calculate the output size of a 28-by-28 image after a 5-by-5 valid convolution with stride 2. Verify it using `correlate2d`. Then change the kernel and explain the resulting map without calling it a learned feature.

**Independent.** Build the MNIST CNN shown in the diagram. Compare it with the dense baseline using the same split, seed policy, epoch budget, and final test contract. Report parameter count, learning curves, confusion matrix, worst-confused digit pair, and inference time.

**Extension: Fashion-MNIST.** State a scientific question about which garment classes share visual ambiguity. Train no more than two justified candidates, inspect class-specific errors and confidence, and document what evidence would be needed before deployment.

**Engineering extension.** Write a tested data loader that validates shape, dtype, range, class vocabulary, and duplicate identifiers. Make a pull request containing the implementation, unit tests, and a brief model-card update.

In [ ]:
# Replace the prediction only after doing the output-shape calculation by hand.
expected_side = (28 - 5) // 2 + 1
probe = correlate2d(np.zeros((28, 28)), np.ones((5, 5)), stride=2)
assert expected_side == 12
assert probe.shape == (expected_side, expected_side)
probe.shape

## Retrieval practice

1. What distinction separates a tensor from a variable?
2. Why is automatic differentiation not an automatic modeling system?
3. What structure does flattening discard?
4. How do stride and padding affect output shape?
5. Why does a CNN have fewer parameters than a dense layer over the same image?
6. Which metrics diagnose optimization, and which evaluate the final decision rule?

## Takeaway

Keras makes MNIST training concise because it packages a lifecycle around TensorFlow operations and automatic differentiation. The important intellectual work remains visible: tensor and label contracts, leakage-safe evaluation, the spatial inductive bias of convolution, reproducible artifacts, and observable training. A CNN is appropriate for images because it learns local shared detectors and composes them into wider receptive fields—not because convolution removes the need for experimental judgment.

## Further reading

- [TensorFlow/Keras guide](https://www.tensorflow.org/guide/keras)
- [Convolutional neural network tutorial](https://www.tensorflow.org/tutorials/images/cnn)
- [Training and evaluation](https://www.tensorflow.org/guide/keras/training_with_built_in_methods)
- [TensorBoard callback API](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/TensorBoard)
- [Automatic differentiation](https://www.tensorflow.org/guide/autodiff)